To do ***Ray Tracing: The Next Week*** in Python and actually *learn* it (not just copy code), build it step-by-step.

The goal is:

$$
L_o(p,\omega_o)=L_e(p,\omega_o)+\int_{\Omega} f_r(p,\omega_i,\omega_o)L_i(p,\omega_i)(n\cdot \omega_i)d\omega_i
$$

This is the **Rendering Equation**.

Where:

| Symbol     | Meaning            | Purpose                |
| ---------- | ------------------ | ---------------------- |
| $L_o$      | outgoing light     | final pixel color      |
| $L_e$      | emitted light      | light source itself    |
| $L_i$      | incoming light     | light hitting object   |
| $f_r$      | BRDF               | material behavior      |
| $n$        | surface normal     | angle effect           |
| $\omega_i$ | incoming direction | where light comes from |
| $\omega_o$ | outgoing direction | toward camera          |

---

# Learning Roadmap

| Step | Topic                | Why                 |
| ---- | -------------------- | ------------------- |
| 1    | Core math            | vectors, dot, cross |
| 2    | Rays                 | shoot through world |
| 3    | Sphere intersections | basic geometry      |
| 4    | Materials            | diffuse/metal/glass |
| 5    | Textures             | image + procedural  |
| 6    | Rectangles           | walls, lights       |
| 7    | Volumes              | fog, smoke          |
| 8    | Transformations      | rotate/move         |
| 9    | BVH                  | speed up            |
| 10   | Cornell Box          | final scene         |

---

# Step 1: Vector Math

Formula:

$$
\vec{a}=(x,y,z)
$$

Length:

$$
|\vec{a}|=\sqrt{x^2+y^2+z^2}
$$

Dot product:

$$
\vec{a}\cdot\vec{b}=x_1x_2+y_1y_2+z_1z_2
$$

Purpose:

* measure angle
* lighting
* reflection

Python:

```python
import numpy as np

class Vec3:
    def __init__(self,x,y,z):
        self.e=np.array([x,y,z],dtype=float)

    def __add__(self,other):
        return Vec3(*(self.e+other.e))

    def __sub__(self,other):
        return Vec3(*(self.e-other.e))

    def __mul__(self,t):
        return Vec3(*(self.e*t))

    def dot(self,other):
        return np.dot(self.e,other.e)

    def length(self):
        return np.linalg.norm(self.e)
```

---

# Step 2: Ray

Formula:

$$
P(t)=A+tB
$$

Where:

| Symbol | Meaning   |
| ------ | --------- |
| $A$    | origin    |
| $B$    | direction |
| $t$    | distance  |

Purpose:

Ray moves in line.

Python:

```python
class Ray:
    def __init__(self,origin,direction):
        self.origin=origin
        self.direction=direction

    def at(self,t):
        return self.origin + self.direction*t
```

---

# Step 3: Sphere Intersection

Sphere equation:

$$
(P-C)\cdot(P-C)=r^2
$$

Substitute ray:

$$
(A+tB-C)\cdot(A+tB-C)=r^2
$$

Quadratic:

$$
at^2+bt+c=0
$$

Where:

$$
a=B\cdot B
$$

$$
b=2B\cdot(A-C)
$$

$$
c=(A-C)\cdot(A-C)-r^2
$$

Solve:

$$
t=\frac{-b\pm\sqrt{b^2-4ac}}{2a}
$$

Purpose:

Find hit point.

Python:

```python
def hit_sphere(center,radius,ray):
    oc=ray.origin-center
    a=ray.direction.dot(ray.direction)
    b=2.0*oc.dot(ray.direction)
    c=oc.dot(oc)-radius*radius
    discriminant=b*b-4*a*c

    if discriminant<0:
        return None

    return (-b-np.sqrt(discriminant))/(2*a)
```

---

# Step 4: Lambertian (Diffuse)

Formula:

$$
f_r=\frac{\rho}{\pi}
$$

Purpose:

Random scatter.

Random hemisphere:

$$
n+\text{random_unit_vector}
$$

Python:

```python
def random_unit():
    while True:
        p=np.random.uniform(-1,1,3)
        if np.linalg.norm(p)<1:
            return p/np.linalg.norm(p)
```

---

# Step 5: Reflection

Formula:

$$
R=V-2(V\cdot N)N
$$

Purpose:

Mirror bounce.

Python:

```python
def reflect(v,n):
    return v - 2*np.dot(v,n)*n
```

---

# Step 6: Refraction

Snell's law:

$$
\eta_i\sin\theta_i=\eta_t\sin\theta_t
$$

Vector form:

$$
R'=\eta (uv+\cos\theta n)-n\sqrt{1-\eta^2(1-\cos^2\theta)}
$$

Purpose:

Glass.

Python:

```python
def refract(uv,n,eta):
    cos_theta=min(np.dot(-uv,n),1.0)
    r_out_perp=eta*(uv+cos_theta*n)
    r_out_parallel=-np.sqrt(abs(1.0-np.dot(r_out_perp,r_out_perp)))*n
    return r_out_perp+r_out_parallel
```

---

# Step 7: Texture Mapping

UV mapping:

$$
u=\frac{\phi}{2\pi}
$$

$$
v=\frac{\theta}{\pi}
$$

Where:

$$
\phi=\tan^{-1}(z,x)
$$

$$
\theta=\sin^{-1}(y)
$$

Purpose:

Map image onto object.

---

# Step 8: Perlin Noise

Interpolation:

$$
f(t)=3t^2-2t^3
$$

Purpose:

Smooth randomness.

Used for:

* marble
* wood
* clouds

---

# Step 9: Rectangles

XY plane:

$$
z=k
$$

Check:

$$
x_0<x<x_1
$$

$$
y_0<y<y_1
$$

Purpose:

Walls, floor, lights.

---

# Step 10: Volumes (Fog)

Beer-Lambert law:

$$
T=e^{-\sigma t}
$$

Where:

| Symbol   | Meaning  |
| -------- | -------- |
| $\sigma$ | density  |
| $t$      | distance |

Purpose:

Light absorption.

Probability:

$$
P=1-e^{-\sigma t}
$$

Used in:

* smoke
* fog
* clouds

---

# Step 11: Transformations

Translation:

$$
P'=P+T
$$

Rotation Y:

$$
x'=x\cos\theta+z\sin\theta
$$

$$
z'=-x\sin\theta+z\cos\theta
$$

Purpose:

Move objects.

---

# Step 12: BVH

Bounding box test:

$$
t_{min}=\frac{x_{min}-o_x}{d_x}
$$

$$
t_{max}=\frac{x_{max}-o_x}{d_x}
$$

Purpose:

Avoid testing every object.

Without BVH:

$$
O(n)
$$

With BVH:

$$
O(\log n)
$$

Big speedup.

---

# Final Pipeline

| Stage      | Formula         | Purpose       |
| ---------- | --------------- | ------------- |
| Camera     | $P(t)=A+tB$     | generate rays |
| Hit        | quadratic       | find object   |
| Normal     | $\frac{P-C}{r}$ | lighting      |
| Material   | BRDF            | bounce        |
| Reflection | $R=V-2(V·N)N$   | metal         |
| Refraction | Snell           | glass         |
| Texture    | UV map          | detail        |
| Volume     | $e^{-\sigma t}$ | fog           |
| BVH        | AABB            | speed         |

---

# Final Cornell Box

Scene:

* 6 rectangles
* 2 boxes
* light source
* smoke volume

This gives realistic:

* global illumination
* shadows
* soft light
* reflections
* volumetric scattering

---

Best learning order:

| Day | Learn           |
| --- | --------------- |
| 1   | vector + ray    |
| 2   | sphere + camera |
| 3   | materials       |
| 4   | textures        |
| 5   | rectangles      |
| 6   | volumes         |
| 7   | BVH             |

This matches the book structure of Ray Tracing in One Weekend and builds toward a full physically-based renderer.
